### Filtrado y limpieza de los datos

In [ ]:
import os

from preprocessing_EEG import extract_labeled_events, preprocessing_grandchamp_v2

bids_root = "C:\\Users\\aadel\\Desktop\\GCID\\Cuarto\\Segundo Cuatrimestre\\TFG\\data\\ds001787-download"

stage = 2

if stage == 3:
    epochs, ica = preprocessing_grandchamp_v2(
        sub=1, stage=3, session=1, bids_root=bids_root, fast_mode=True
    )
else:
    epochs = preprocessing_grandchamp_v2(
        sub=1, stage=stage, session=1, bids_root=bids_root, fast_mode=True
    )

### Extracción de características para SVM

In [7]:
from components_extraction import extract_features_from_epochs
import mne

path_input = 'C:\\Users\\aadel\\Desktop\\GCID\\Cuarto\\Segundo Cuatrimestre\\TFG\\python\\preprocessing\\001_01_epochs_ica_a2-epo.fif'
epochs = mne.read_epochs(path_input, preload=True)

# 2. Extraer características
# Nota: El script usará PO7, Pz, PO8 y Fz (mapeados a A10, A19, B7, C21)
df = extract_features_from_epochs(
    epochs,
    baseline_window=(-10, -9), # Primer segundo de la época
    signal_window=(-1, 0)      # Último segundo antes del aviso
)

# 3. Guardar resultados
df.to_csv('features_MW_study.csv', index=False)
print("\n" + "="*30)
print(f"¡ÉXITO! Archivo guardado como 'features_MW_study.csv'")
print(f"Dimensiones de la matriz: {df.shape}")

# 4. Análisis rápido de control (¿Funciona nuestra hipótesis?)
print("\nPROMEDIOS POR ESTADO (ALPHA EN Pz):")
resumen = df.groupby('label')['alpha_power_Pz_signal'].mean()
print(resumen)
print("="*30)

Reading C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing\001_01_epochs_ica_a2-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =  -10000.00 ...       0.00 ms
        0 CTF compensation matrices available
Adding metadata with 14 columns
25 matching events found
No baseline correction applied
0 projection items activated

FEATURE EXTRACTION
Epochs: 25
Channels: ['PO7', 'Pz', 'PO8', 'Fz']
Alpha band: 8.5-12 Hz
Theta band: 4-8 Hz
Baseline window: -10 to -9 s
Signal window: -1 to 0 s
⚠️  Warning: Channel Fz/C21 not found

✓ Using channels: {'PO7': 'A10', 'Pz': 'A19', 'PO8': 'B7'}
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).

Baseline samples: 257
Signal samples: 257

--- Creating Filters ---
⚠️  Warning: Filter SSE = 39.1677 (should be < 1.0)
   Consider increasing filter length or adjusting parameters
⚠️  Warning: Filter SSE = 28.6530 (should be < 1.0)
   Consider increasing filter length or adjusting

### Aplicación de ISPC

In [2]:
import numpy as np
from components_extraction import compute_ispc
# 1. Configuración básica
sfreq = 256                  # Frecuencia de muestreo (Hz)
t = np.arange(0, 1, 1/sfreq) # Vector de tiempo de 1 segundo
freq = 10                    # Frecuencia de la señal (10 Hz - Alpha)

# 2. Creamos la señal X (Referencia)
# Una señal analítica es exp(i * 2 * pi * f * t)
analytic_x = np.exp(1j * 2 * np.pi * freq * t)

# 3. Creamos la señal Y (Con desfase variable)
# Le añadimos un desfase inicial (pi/4) y un poco de ruido aleatorio
# para que la sincronía no sea perfecta (ISPC < 1.0)
ruido_fase = np.random.normal(0, 0.5, size=t.shape) # Ruido de fase
analytic_y = np.exp(1j * (2 * np.pi * freq * t + np.pi/4 + ruido_fase))

result = compute_ispc(analytic_x, analytic_y)
print(result)

0.8827537978067765


### Pipeline
#### Preprocesado

In [1]:
import os
import sys
from pathlib import Path
from preprocessing_EEG import preprocessing_grandchamp_v2

# --- CLASE LOGGER PARA REDIRECCIÓN ---
class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# --- CONFIGURACIÓN ---
bids_root = "C:\\Users\\aadel\\Desktop\\GCID\\Cuarto\\Segundo Cuatrimestre\\TFG\\clean_data"
# Aseguramos que exista una carpeta para los logs
log_dir = Path("logs_ejecucion")
log_dir.mkdir(exist_ok=True)

subs = len(os.listdir("../clean_data"))
stages = [1, 3]
sessions = [1, 2]

# --- BUCLE PRINCIPAL ---
for i in range(1, subs + 1):
    for session in sessions:
        sub_str = f"{i:03d}"
        # Definimos el archivo de log para este sujeto
        log_file = log_dir / f"log_sub-{sub_str}-sess-{session}.txt"

        # Iniciamos la redirección
        original_stdout = sys.stdout
        sys.stdout = Logger(log_file)

        print(f"\n" + "#"*80)
        print(f"INICIANDO PROCESAMIENTO SUJETO: {sub_str}")
        print("#"*80)

        try:
            for stage in stages:
                if stage == 3:
                    epochs, ica = preprocessing_grandchamp_v2(
                        sub=i, stage=3, session=session, bids_root=bids_root, fast_mode=True
                    )
                else:
                    epochs = preprocessing_grandchamp_v2(
                        sub=i, stage=stage, session=session, bids_root=bids_root, fast_mode=True
                    )
        except Exception as e:
            print(f"\n❌ ERROR CRÍTICO en Sujeto {sub_str}: {str(e)}")
        finally:
            # IMPORTANTE: Volvemos a la salida normal antes de pasar al siguiente sujeto
            print(f"\nFinalizado procesamiento Sujeto {sub_str}.")
            sys.stdout.log.close() # Cerramos el archivo de este sujeto
            sys.stdout = original_stdout

print("\n✅ Proceso completo para todos los sujetos. Revisa la carpeta 'logs_ejecucion'.")

Using qt as 2D backend.

################################################################################
INICIANDO PROCESAMIENTO SUJETO: 001
################################################################################

=== Stage 1: Stimulus-Locked Epoching ===
Subject: 001, Session: 01
⚡ FAST MODE enabled

Loading: C:/Users/aadel/Desktop/GCID/Cuarto/Segundo Cuatrimestre/TFG/clean_data/sub-001/ses-01/eeg/sub-001_ses-01_task-meditation_eeg.bdf
Extracting EDF parameters from C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-001\ses-01\eeg\sub-001_ses-01_task-meditation_eeg.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 696575  =      0.000 ...  2720.996 secs...


C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing_EEG.py:259: RuntimeWarning: Did not find any events.tsv associated with sub-001_ses-01_task-Meditation.

The search_str was "C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-001\**\eeg\sub-001_ses-01*events.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=True)
C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing_EEG.py:259: RuntimeWarning: Did not find any channels.tsv associated with sub-001_ses-01_task-Meditation.

The search_str was "C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-001\**\eeg\sub-001_ses-01*channels.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=True)
C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing_EEG.py:259: RuntimeWarning: Did not find any eeg.json associated with sub-001_ses-01_task-Meditation.

The search_str was "C:\Users\aadel\Desktop\GCID\Cuarto\Segun

Channels: 80, Sfreq: 256.0 Hz
Re-referencing to: ['A16', 'B21']
EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.
Filtering: 0.1-42 Hz
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 42 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 42.00 Hz
- Upper transition bandwidth: 10.50 Hz (-6 dB cutoff frequency: 47.25 Hz)
- Filter length: 8449 samples (33.004 s)

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
87 events found on stim channel Status
Event IDs: [  2   4 128]

Tot

C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing_EEG.py:459: RuntimeWarning: The epochs you passed to ICA.fit() were baseline-corrected. However, we suggest to fit ICA only on data that has been high-pass filtered, but NOT baseline-corrected.
  ica.fit(epochs)


Selecting by number: 15 components
Fitting ICA took 2.3s.
Writing ICA solution to C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing\session_1\001_01_ica.fif...

✓ Saved: 001_01_ica.fif, 001_01_epochs_ica-epo.fif

Finalizado procesamiento Sujeto 001.

################################################################################
INICIANDO PROCESAMIENTO SUJETO: 001
################################################################################

=== Stage 1: Stimulus-Locked Epoching ===
Subject: 001, Session: 02
⚡ FAST MODE enabled

Loading: C:/Users/aadel/Desktop/GCID/Cuarto/Segundo Cuatrimestre/TFG/clean_data/sub-001/ses-02/eeg/sub-001_ses-02_task-meditation_eeg.bdf
Extracting EDF parameters from C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-001\ses-02\eeg\sub-001_ses-02_task-meditation_eeg.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 695551  =      0.000 ...  2716.9

...

C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing_EEG.py:259: RuntimeWarning: Did not find any events.tsv associated with sub-024_ses-01_task-Meditation.

The search_str was "C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-024\**\eeg\sub-024_ses-01*events.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=True)
C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing_EEG.py:259: RuntimeWarning: Did not find any channels.tsv associated with sub-024_ses-01_task-Meditation.

The search_str was "C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-024\**\eeg\sub-024_ses-01*channels.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=True)
C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing_EEG.py:259: RuntimeWarning: Did not find any eeg.json associated with sub-024_ses-01_task-Meditation.

The search_str was "C:\Users\aadel\Desktop\GCID\Cuarto\Segun

Channels: 80, Sfreq: 256.0 Hz
Re-referencing to: ['A16', 'B21']
EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.
Filtering: 0.1-42 Hz
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 42 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 42.00 Hz
- Upper transition bandwidth: 10.50 Hz (-6 dB cutoff frequency: 47.25 Hz)
- Filter length: 8449 samples (33.004 s)

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
164 events found on stim channel Status
Event IDs: [  2   4   8 128]

C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing_EEG.py:459: RuntimeWarning: The epochs you passed to ICA.fit() were baseline-corrected. However, we suggest to fit ICA only on data that has been high-pass filtered, but NOT baseline-corrected.
  ica.fit(epochs)


Selecting by number: 15 components
Fitting ICA took 1.5s.
Writing ICA solution to C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing\session_1\024_01_ica.fif...

✓ Saved: 024_01_ica.fif, 024_01_epochs_ica-epo.fif

Finalizado procesamiento Sujeto 024.

################################################################################
INICIANDO PROCESAMIENTO SUJETO: 024
################################################################################

=== Stage 1: Stimulus-Locked Epoching ===
Subject: 024, Session: 02
⚡ FAST MODE enabled

❌ ERROR CRÍTICO en Sujeto 024: [WinError 3] El sistema no puede encontrar la ruta especificada: 'C:\\Users\\aadel\\Desktop\\GCID\\Cuarto\\Segundo Cuatrimestre\\TFG\\clean_data\\sub-024\\ses-02'

Finalizado procesamiento Sujeto 024.

✅ Proceso completo para todos los sujetos. Revisa la carpeta 'logs_ejecucion'.


C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing_EEG.py:459: RuntimeWarning: Using n_components=15 (resulting in n_components_=15) may lead to an unstable mixing matrix estimation because the ratio between the largest (56) and smallest (3.6e-11) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 1
  ica.fit(epochs)


### Aplicación de CWT para compatibilidad con CNN

In [ ]:
import mne
import numpy as np
from mne.time_frequency import tfr_morlet


epochs = mne.read_epochs('preprocessing/001_01_epochs_ica_a2-epo.fif')

channels = ['A10', 'A19', 'B7'] # PO7, Pz, PO8, Fz
epochs.pick_channels(channels)

freqs = np.linspace(4, 40, num=40)

n_cycles = freqs / 2.

tfr = tfr_morlet(epochs, freqs=freqs, n_cycles=n_cycles,
                 return_itc=False, average=False)

x_data = tfr.data

x_data = np.abs(x_data)**2

x_data = np.log10(x_data)

y_labels = epochs.events[:, 2] # 0 para On-Task, 1 para Mind Wandering

print(f"Formato final para la CNN (X): {x_data.shape}")
print(f"Formato de etiquetas (y): {y_labels.shape}")
import numpy as np
np.savez_compressed("ml_data/001_001_cwt_data.npz", x=x_data, y=y_labels)

In [1]:
import numpy as np
import mne
from mne.time_frequency import tfr_morlet

def cwt_application(file, session, subject):

    path = f'./preprocessing/session_{session}/' + file
    epochs = mne.read_epochs(path)

    channels = ['A10', 'A19', 'B7'] # PO7, Pz, PO8, Fz
    epochs.pick_channels(channels)

    times = epochs.times
    ch_names = epochs.ch_names
    freqs = np.linspace(4, 40, num=40)

    n_cycles = freqs / 2.

    tfr = tfr_morlet(epochs, freqs=freqs, n_cycles=n_cycles,
                     return_itc=False, average=False)

    x_data = tfr.data

    x_data = np.abs(x_data)**2

    x_data = np.log10(x_data)

    y_labels = epochs.events[:, 2] # 0 para On-Task, 1 para Mind Wandering

    print(f"Formato final para la CNN (X): {x_data.shape}")
    print(f"Formato de etiquetas (y): {y_labels.shape}")
    np.savez_compressed(f'ml_data/0{subject}_00{session}_cwt_data.npz', power=x_data, label=y_labels, frex = freqs, times=times, ch_names=ch_names)

In [3]:
import re
import os
sessions = [1,2]
for session in sessions:
    files = os.listdir(f'preprocessing/session_{session}')
    patron = r".*ica-epo\.fif$"
    t = re.compile(patron)


    for file in files:
        match = t.search(file)
        if match:
            if session == 1:
                sub = file[1:3]
                cwt_application(file, 1, sub)
            else:
                sub = file[1:3]
                cwt_application(file, 2, sub)

Reading C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing\session_1\001_01_epochs_ica-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =  -10000.00 ...       0.00 ms
        0 CTF compensation matrices available
Adding metadata with 14 columns
26 matching events found
No baseline correction applied
0 projection items activated
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
Formato final para la CNN (X): (26, 3, 40, 2561)
Formato de etiquetas (y): (26,)
Reading C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\python\preprocessing\session_1\002_01_epochs_ica-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =  -10000.00 ...       0.00 ms
        0 CTF compensation matrices available
Adding metadata with 14 columns
31 matching events found
No baseline correction applied
0 projecti

In [2]:
import re

sessions = [1,2]
epochs_session_1 = []
epochs_session_2 = []

for session in sessions:
    files = os.listdir(f'preprocessing/session_{session}')
    patron = r".*ica-epo\.fif$"
    t = re.compile(patron)


    for file in files:
        match = t.search(file)
        if match:
            if session == 1:
                epochs_session_1.append(file)
            else:
                epochs_session_2.append(file)

print(len(epochs_session_1))
print(len(epochs_session_2))
print(epochs_session_1)

21
14
['001_01_epochs_ica-epo.fif', '002_01_epochs_ica-epo.fif', '003_01_epochs_ica-epo.fif', '004_01_epochs_ica-epo.fif', '005_01_epochs_ica-epo.fif', '006_01_epochs_ica-epo.fif', '007_01_epochs_ica-epo.fif', '008_01_epochs_ica-epo.fif', '010_01_epochs_ica-epo.fif', '011_01_epochs_ica-epo.fif', '012_01_epochs_ica-epo.fif', '013_01_epochs_ica-epo.fif', '015_01_epochs_ica-epo.fif', '016_01_epochs_ica-epo.fif', '018_01_epochs_ica-epo.fif', '019_01_epochs_ica-epo.fif', '020_01_epochs_ica-epo.fif', '021_01_epochs_ica-epo.fif', '022_01_epochs_ica-epo.fif', '023_01_epochs_ica-epo.fif', '024_01_epochs_ica-epo.fif']


In [22]:
print(epochs_session_1)

['001_01_epochs_ica-epo.fif', '002_01_epochs_ica-epo.fif', '003_01_epochs_ica-epo.fif', '004_01_epochs_ica-epo.fif', '005_01_epochs_ica-epo.fif', '006_01_epochs_ica-epo.fif', '007_01_epochs_ica-epo.fif', '008_01_epochs_ica-epo.fif', '010_01_epochs_ica-epo.fif', '011_01_epochs_ica-epo.fif', '012_01_epochs_ica-epo.fif', '013_01_epochs_ica-epo.fif', '015_01_epochs_ica-epo.fif', '016_01_epochs_ica-epo.fif', '018_01_epochs_ica-epo.fif', '019_01_epochs_ica-epo.fif', '020_01_epochs_ica-epo.fif', '021_01_epochs_ica-epo.fif', '022_01_epochs_ica-epo.fif', '023_01_epochs_ica-epo.fif', '024_01_epochs_ica-epo.fif']


In [27]:
from mne_bids import BIDSPath, read_raw_bids
import numpy as np

from preprocessing_EEG import extract_labeled_events
import mne
bids_root = "C:\\Users\\aadel\\Desktop\\GCID\\Cuarto\\Segundo Cuatrimestre\\TFG\\clean_data"
bids_path = BIDSPath(subject="009", session="01", task="Meditation", root=bids_root)

raw = read_raw_bids(bids_path=bids_path, verbose=True)
events = mne.find_events(raw, stim_channel='Status', verbose=True)
events_labeled, metadata, report = extract_labeled_events(
            events,
            event_id_stim=128,
            event_ids_response=[2, 4, 8],
            min_responses=1,
            max_response_delay=10.0,
            sfreq=raw.info['sfreq']
        )
unique_labels = np.unique(events_labeled[:, 2])
print(f"DEBUG: Etiquetas encontradas: {unique_labels}")

if 1 not in unique_labels:
    print("⚠️ MOCK: No hay Mind Wandering. Forzando uno en el primer ensayo...")
    events_labeled[0, 2] = 1  # Convertimos el primer evento de 0 a 1
# -----------------
event_dict = {
            'on_task': 0,
            'mind_wandering': 1
        }
# Ahora MNE no debería dar error al buscar 'mind_wandering'
epochs = mne.Epochs(
            raw,
            events_labeled,
            event_id=event_dict,
            tmin=-10,
            tmax=0,
            baseline=(-0.5, 0),
            preload=True,
            verbose=True,
            metadata=metadata
        )
epochs.save("prueba.fif")

Extracting EDF parameters from C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-009\ses-01\eeg\sub-009_ses-01_task-meditation_eeg.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Finding events on: Status


C:\Users\aadel\AppData\Local\Temp\ipykernel_10012\2456602531.py:9: RuntimeWarning: Did not find any events.tsv associated with sub-009_ses-01_task-Meditation.

The search_str was "C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-009\**\eeg\sub-009_ses-01*events.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=True)
C:\Users\aadel\AppData\Local\Temp\ipykernel_10012\2456602531.py:9: RuntimeWarning: Did not find any channels.tsv associated with sub-009_ses-01_task-Meditation.

The search_str was "C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-009\**\eeg\sub-009_ses-01*channels.tsv"
  raw = read_raw_bids(bids_path=bids_path, verbose=True)
C:\Users\aadel\AppData\Local\Temp\ipykernel_10012\2456602531.py:9: RuntimeWarning: Did not find any eeg.json associated with sub-009_ses-01_task-Meditation.

The search_str was "C:\Users\aadel\Desktop\GCID\Cuarto\Segundo Cuatrimestre\TFG\clean_data\sub-009\**\eeg\sub-009_ses-01*eeg.json"
  raw =

Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
81 events found on stim channel Status
Event IDs: [  2   4   8 128]

EXTRACTING LABELED EVENTS (Stimulus-Locked Approach)
✓ Epoch center: Trigger 128 (stimulus onset)
✓ Epoch window: [-10, 0] seconds (mental state BEFORE question)
✓ Labels: Based on Q2 response (4 or 8 = Mind Wandering)

Found 32 stimulus events (trigger 128)

✓ Extraction complete:
  Total stimuli: 32
  Included: 32 (100.0%)
  Excluded: 0

  Label distribution:
    on_task: 17 (53.1%)
    unknown: 15 (46.9%)
DEBUG: Etiquetas encontradas: [  0 999]
⚠️ MOCK: No hay Mind Wandering. Forzando uno en el primer ensayo...
Adding metadata with 11 columns
18 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 18 events and 2561 original time points ...
0 bad epochs dropped


C:\Users\aadel\AppData\Local\Temp\ipykernel_10012\2456602531.py:42: RuntimeWarning: This filename (prueba.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs.save("prueba.fif")


[WindowsPath('C:/Users/aadel/Desktop/GCID/Cuarto/Segundo Cuatrimestre/TFG/python/prueba.fif')]

In [ ]:
from mne_bids import BIDSPath, read_raw_bids
import numpy as np

from preprocessing_EEG import extract_labeled_events
import mne
bids_root = "C:\\Users\\aadel\\Desktop\\GCID\\Cuarto\\Segundo Cuatrimestre\\TFG\\clean_data"
bids_path = BIDSPath(subject="009", session="01", task="Meditation", root=bids_root)

raw = read_raw_bids(bids_path=bids_path, verbose=True)
events = mne.find_events(raw, stim_channel='Status', verbose=True)
events_labeled, metadata, report = extract_labeled_events(
            events,
            event_id_stim=128,
            event_ids_response=[2, 4, 8],
            min_responses=1,
            max_response_delay=10.0,
            sfreq=raw.info['sfreq']
        )
unique_labels = np.unique(events_labeled[:, 2])
print(f"DEBUG: Etiquetas encontradas: {unique_labels}")

if 1 not in unique_labels:
    print("⚠️ MOCK: No hay Mind Wandering. Forzando uno en el primer ensayo...")
    events_labeled[0, 2] = 1  # Convertimos el primer evento de 0 a 1
# -----------------
event_dict = {
            'on_task': 0,
            'mind_wandering': 1
        }
# Ahora MNE no debería dar error al buscar 'mind_wandering'
epochs = mne.Epochs(
            raw,
            events_labeled,
            event_id=event_dict,
            tmin=-10,
            tmax=0,
            baseline=(-0.5, 0),
            preload=True,
            verbose=True,
            metadata=metadata
        )
epochs.save("prueba.fif")

35